In [17]:
import pandas as pd
import numpy as np
import requests
import time
import os
from dotenv import load_dotenv

load_dotenv('../.env')
token = os.getenv('ONEMAP_TOKEN')

# Verify token loaded
print("Token loaded:", token is not None)

df = pd.read_csv('../data/raw/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv')
unique_addresses = df[['block', 'street_name']].drop_duplicates().reset_index(drop=True)

print(f"Total transactions: {len(df):,}")
print(f"Unique addresses:   {len(unique_addresses):,}")

Token loaded: True
Total transactions: 233,055
Unique addresses:   9,712


In [18]:
def geocode_address(block, street_name, token, retries=3):
    query = f"{block} {street_name} SINGAPORE"
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": query,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }
    headers = {
        "Authorization": token
    }
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            data = response.json()
            if data.get('found', 0) > 0:
                result = data['results'][0]
                return float(result['LATITUDE']), float(result['LONGITUDE'])
            else:
                time.sleep(1)
        except Exception as e:
            time.sleep(1)
    return None, None

# Test on one address
block, street = unique_addresses.iloc[0]['block'], unique_addresses.iloc[0]['street_name']
lat, lon = geocode_address(block, street, token)
print(f"Test address: {block} {street}")
print(f"Coordinates:  {lat}, {lon}")

Test address: 406 ANG MO KIO AVE 10
Coordinates:  1.36200453938712, 103.853879910407


In [19]:
test = unique_addresses.head(50).copy().reset_index(drop=True)
results = []

for idx, row in test.iterrows():
    lat, lon = geocode_address(row['block'], row['street_name'], token)
    results.append({
        'block': row['block'],
        'street_name': row['street_name'],
        'latitude': lat,
        'longitude': lon
    })
    time.sleep(0.25)

test_results = pd.DataFrame(results)
print(f"Success: {test_results['latitude'].notna().sum()} / 50")
print(f"Failed:  {test_results['latitude'].isna().sum()} / 50")

Success: 50 / 50
Failed:  0 / 50


In [ ]:
results = []

for idx, row in unique_addresses.iterrows():
    lat, lon = geocode_address(row['block'], row['street_name'], token)
    results.append({
        'block': row['block'],
        'street_name': row['street_name'],
        'latitude': lat,
        'longitude': lon
    })
    
    if (idx + 1) % 500 == 0:
        print(f"Processed {idx + 1:,} / {len(unique_addresses):,}")
        pd.DataFrame(results).to_csv('../data/processed/geocoded_addresses.csv', index=False)
    
    time.sleep(0.25)

geocoded = pd.DataFrame(results)
geocoded.to_csv('../data/processed/geocoded_addresses.csv', index=False)
print(f"Done. Geocoded {len(geocoded):,} addresses.")

Processed 500 / 9,712
